### A5.1.15. Reduce Lock Contention in Class Design

**Problem:**

Given a class that uses a single coarse-grained lock for all operations, redesign it to reduce lock contention so multiple threads can make progress simultaneously.

**Explanation:**

Lock contention happens when multiple threads compete for the same lock, causing them to wait instead of doing useful work. The fix is to reduce the scope of locking: protect only what needs protection, and use separate locks for independent pieces of state.

How to reduce contention:

1. Identify independent pieces of state within the class.
2. Replace the single global lock with one lock per independent piece of state (fine-grained locking).
3. Each method acquires only the lock for the state it touches.
4. Operations on different state pieces can now proceed in parallel.

**Example:**

A `UserStore` with a single lock for both `users` and `sessions` dicts. Split into two locks: `users_lock` protects `users`, `sessions_lock` protects `sessions`. Now adding a user and creating a session can happen concurrently.

In [ ]:
import threading
import time


class UserStoreCoarse:
    def __init__(self):
        self.users = {}
        self.sessions = {}
        self.lock = threading.Lock()

    def add_user(self, user_id, name):
        with self.lock:
            self.users[user_id] = name

    def create_session(self, session_id, user_id):
        with self.lock:
            self.sessions[session_id] = user_id


class UserStoreFineGrained:
    def __init__(self):
        self.users = {}
        self.sessions = {}
        self.users_lock = threading.Lock()
        self.sessions_lock = threading.Lock()

    def add_user(self, user_id, name):
        with self.users_lock:
            self.users[user_id] = name

    def create_session(self, session_id, user_id):
        with self.sessions_lock:
            self.sessions[session_id] = user_id


def benchmark(store_class, label):
    store = store_class()
    num_operations = 50000

    def add_users():
        for index in range(num_operations):
            store.add_user(index, f"user_{index}")

    def add_sessions():
        for index in range(num_operations):
            store.create_session(index, index)

    start = time.perf_counter()

    user_thread = threading.Thread(target=add_users)
    session_thread = threading.Thread(target=add_sessions)

    user_thread.start()
    session_thread.start()

    user_thread.join()
    session_thread.join()

    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.4f}s — {len(store.users)} users, {len(store.sessions)} sessions")


benchmark(UserStoreCoarse, "Coarse-grained lock")
benchmark(UserStoreFineGrained, "Fine-grained locks")

**References:**

📘 Herlihy, M. & Shavit, N. (2012). *The Art of Multiprocessor Programming* — Fine-Grained Synchronization

📘 Python Documentation — [threading.Lock](https://docs.python.org/3/library/threading.html#lock-objects)

---

[⬅️ Previous: Fix Data Race in Class Methods](./14_fix_data_race_in_class_methods.ipynb)